# Starter smoke test

Run cells top to bottom with this project's **.venv** Python kernel (Python 3.11).
From the repo root: `uv sync --locked --group notebook`, then
`uv run --locked --group notebook jupyter notebook`.

Local imports, CLI and unit tests run without AWS or database credentials.
To check databases, fill in the repo's ignored `.env`, authenticate your AWS
profile (SSO only if applicable), connect VPN if needed, then set
`RUN_DATABASE_CHECKS = True` below and **restart kernel → Run All**.

Database checks read connection metadata only. Shared table discovery reads
`information_schema`, not business rows. Shared DAL defaults to a `dbo` secret;
read-only sessions are a guardrail, not database-enforced read-only grants.
Never save secrets or data outputs in Git. Clear all outputs before committing.

In [ ]:
import os
import subprocess
import sys
from contextlib import AbstractContextManager
from importlib.metadata import version
from pathlib import Path
from typing import cast

from aily_data_access_layer.config.database import DBConfig
from aily_data_access_layer.connectors.db.db import DBConnector
from aily_data_access_layer.dal import Dal
from aily_data_access_layer.utils.db_type import DBTypes
from aily_py_commons.aily_settings import AilySettings
from click.testing import CliRunner
from dotenv import load_dotenv

from aily_project.cli import cli
from aily_project.services.database.service import (
    check_connection,
    check_shared_connection,
)

# Works when Jupyter starts in the repo root or notebooks/ directory.
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
assert (project_root / "pyproject.toml").is_file(), "Start Jupyter in the repo."
assert Path(sys.prefix).resolve() == (project_root / ".venv").resolve(), (
    "Select this project's .venv kernel, then restart and Run All."
)
assert sys.version_info[:2] == (3, 11), "This starter requires Python 3.11."

results: dict[str, str] = {"kernel": "PASS"}
for package in ("aily-project-starter", "aily-data-access-layer", "aily-py-commons"):
    print(f"{package}: {version(package)}")
results["imports"] = "PASS"
print("PASS: project kernel and published Aily imports.")

## Local behavior

These checks use mocks, never cloud clients or a real database.

In [ ]:
help_result = CliRunner().invoke(cli, ["--help"])
assert help_result.exit_code == 0, "CLI help failed."
assert "database" in help_result.output, "Database commands are missing."
results["CLI"] = "PASS"
print(help_result.output)

test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-m", "unit", "-q"],
    cwd=project_root,
    capture_output=True,
    text=True,
    timeout=120,
    check=False,
)
# Unit test output contains no live database or credential results.
print(test_result.stdout)
assert test_result.returncode == 0, "Unit tests failed; run uv run pytest in terminal."
results["unit tests"] = "PASS"

## Runtime context and optional database access

Toggle database access below. Package-index auth and runtime AWS/DB auth are
separate. Shell variables override `.env`; no parent repo's `.env` is loaded.
Restart the kernel after changing tenant, environment or profile.

In [ ]:
RUN_DATABASE_CHECKS = False
DISCOVER_SHARED_TABLE = True
TARGET_TABLE = "social_media_bright_data_instagram_posts"

load_dotenv(project_root / ".env", override=False)
required = ("AILY_TENANT", "AILY_ENV", "AILY_CLOUD_PROVIDER")
missing = [name for name in required if not os.environ.get(name, "").strip()]
print("Runtime context:", "MISSING " + ", ".join(missing) if missing else "SET")
print("AWS profile:", "SET" if os.environ.get("AWS_PROFILE") else "default chain")
print("AWS region:", "SET" if os.environ.get("AWS_REGION") else "default chain")
print("Database checks:", "ENABLED" if RUN_DATABASE_CHECKS else "SKIPPED")
# Do not print .env, settings objects, DAL objects or raw credential exceptions.

In [ ]:
if RUN_DATABASE_CHECKS:
    if missing:
        raise RuntimeError("Fill .env before enabling DB checks: " + ", ".join(missing))
    if "dal" in globals():
        raise RuntimeError(
            "Restart kernel before initializing another runtime context."
        )
    try:
        AilySettings(use_dotenv=False)
        dal = Dal(
            db_config=DBConfig(
                db_user="ro",
                session_settings={"default_transaction_read_only": "on"},
            ),
            shared_db_config=DBConfig(
                db_type=DBTypes.AURORA,
                db_user="dbo",
                session_settings={"default_transaction_read_only": "on"},
            ),
        )
    except Exception as exc:
        results["runtime initialization"] = f"FAIL ({type(exc).__name__})"
        raise RuntimeError(
            "Aily runtime initialization failed. Check .env and cloud configuration."
        ) from None
    results["runtime initialization"] = "PASS"
    print("PASS: AilySettings initialized before DAL.")
else:
    results["runtime initialization"] = "SKIP (database checks disabled)"

## Tenant and shared connection checks

Each connector is closed after checking. Tenant uses `ro`; shared uses DAL's
shared-secret route (or `DAL_SHARED_DB_CREDENTIALS_PATH` override).
Failures are reported separately, so lacking tenant access does not hide the
shared check. Inspect terminal diagnostics privately if more detail is needed.

In [ ]:
if RUN_DATABASE_CHECKS:
    for name, check in (
        ("tenant DB", check_connection),
        ("shared DB", check_shared_connection),
    ):
        try:
            info = check(dal)
            if str(info.get("read_only", "")).lower() != "on":
                raise RuntimeError("Connection is not read-only.")
        except Exception as exc:
            results[name] = f"FAIL ({type(exc).__name__})"
            print(
                f"{name}: {results[name]}. Check AWS login, secret permissions, "
                "VPN/network and database grants. No credentials were printed."
            )
        else:
            results[name] = "PASS"
            print(f"{name}: PASS (connected, transaction_read_only=on)")
else:
    results["tenant DB"] = "SKIP"
    results["shared DB"] = "SKIP"

## Find the Instagram table in the shared DB

This optional query returns up to 20 matching schema/table names only.
It does not retrieve posts. No match means the table is absent or invisible to
this credential; it is not a connection failure.

In [ ]:
if RUN_DATABASE_CHECKS and DISCOVER_SHARED_TABLE and results.get("shared DB") == "PASS":
    try:
        with cast(AbstractContextManager[DBConnector], dal.shared_db) as shared_db:
            status = shared_db.execute_and_fetch(
                "SELECT current_setting('transaction_read_only') AS read_only"
            )[0]
            if status["read_only"] != "on":
                raise RuntimeError("Refusing discovery on a writable session.")
            tables = shared_db.execute_and_fetch(
                """
                SELECT table_schema, table_name
                FROM information_schema.tables
                WHERE table_name = %s
                ORDER BY table_schema, table_name
                LIMIT 20
                """,
                bind_params=(TARGET_TABLE,),
            )
        results["shared table discovery"] = "PASS" if tables else "NOT FOUND"
        for table in tables:
            print(f"{table['table_schema']}.{table['table_name']}")
        if not tables:
            print("No visible match. Confirm schema/location and shared DB grants.")
    except Exception as exc:
        results["shared table discovery"] = f"FAIL ({type(exc).__name__})"
        print("Discovery failed. Check permissions; raw exceptions are not printed.")
else:
    results["shared table discovery"] = "SKIP"

## Summary

`SKIP` is not proof of cloud/database access. `NOT FOUND` is not proof the table
does not exist. Only `PASS` connection checks confirm that access in this runtime.
Clear notebook outputs before committing.

In [ ]:
for name, status in results.items():
    print(f"{name:28} {status}")
if not RUN_DATABASE_CHECKS:
    print("\nLocal checks only. Set RUN_DATABASE_CHECKS=True to verify DB access.")
elif any(status.startswith("FAIL") for status in results.values()):
    print("\nSome checks failed. Resolve the indicated layer and restart Run All.")
else:
    print("\nEnabled checks completed; review any SKIP or NOT FOUND above.")